In [1]:
!pip install -q huggingface_hub rich

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box
from rich.rule import Rule
from huggingface_hub import InferenceClient
from kaggle_secrets import UserSecretsClient
from datetime import date
import json

# Load HF Token
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

# Use Groq provider — free and fast
client = InferenceClient(api_key=HF_TOKEN)

console = Console()

console.print()
console.print(Panel.fit(
    "[bold cyan]  🎓 E X A M E A S E  A I  A G E N T  v2.0  [/bold cyan]\n"
    "[dim]  Your Personal Exam Survival Concierge  [/dim]\n"
    "[dim]  Powered by Hugging Face + Groq  [/dim]",
    border_style="cyan",
    box=box.DOUBLE,
    padding=(1, 4)
))
console.print()
console.print("[bold green]✅ Setup complete! ExamEase is ready.[/bold green]\n")

╔════════════════════════════════════════════════════╗
║                                                    ║
║      🎓 E X A M E A S E  A I  A G E N T  v2.0      ║
║      Your Personal Exam Survival Concierge         ║
║      Powered by Hugging Face + Groq                ║
║                                                    ║
╚════════════════════════════════════════════════════╝

✅ Setup complete! ExamEase is ready.

In [2]:
# ── Agent Memory (State) ─────────────────────────────────────
agent_memory = {
    "student_name": "",
    "subjects":      [],
    "studied_topics": {},
    "session_log":   []
}

# ── System Prompt (SKILL.md style) ───────────────────────────
SYSTEM_PROMPT = """You are ExamEase, an intelligent and friendly personal exam survival concierge agent for students.

Your core skills:
1. SCHEDULE  — Create realistic day-by-day study plans based on exam dates and available hours
2. EXPLAIN   — Break down complex academic topics into simple, clear explanations
3. MOTIVATE  — Provide calm, focused encouragement when students feel stressed
4. QA        — Answer subject-specific academic questions accurately and concisely

Rules:
- Always be warm, practical, and encouraging
- Keep responses focused and not too long — students are pressed for time
- Format schedules clearly with dates, subjects, and hours
- Use bullet points and structure for clarity
- Never overwhelm — prioritize what matters most"""

# ── Core Ask Function ─────────────────────────────────────────
def ask_agent(user_message):
    context = f"""
Student: {agent_memory['student_name']}
Today: {date.today().strftime('%A, %B %d, %Y')}
Subjects & Exams: {json.dumps(agent_memory['subjects'], indent=2)}
Already Studied: {json.dumps(agent_memory['studied_topics'], indent=2)}

Student says: {user_message}
"""
    console.print(f"\n[bold white]💬 You:[/bold white] [italic white]{user_message}[/italic white]")
    console.print("[dim cyan]⏳ ExamEase is thinking...[/dim cyan]")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": context}
    ]

    try:
        completion = client.chat.completions.create(
            model="meta-llama/Llama-4-Scout-17B-16E-Instruct",
            messages=messages,
            max_tokens=600,
            temperature=0.7,
        )
        reply = completion.choices[0].message.content.strip()
    except Exception as e:
        reply = f"⚠️ Could not reach model: {str(e)[:120]}"

    # Log
    agent_memory["session_log"].append({
        "user":  user_message[:60],
        "agent": reply[:120]
    })

    console.print(Panel(
        f"[white]{reply}[/white]",
        title="[bold cyan]🤖 ExamEase[/bold cyan]",
        border_style="cyan",
        box=box.ROUNDED,
        padding=(1, 2)
    ))
    return reply

# ── Setup Student Profile ─────────────────────────────────────
def setup_student(name, subjects_info):
    agent_memory["student_name"] = name
    agent_memory["subjects"]     = subjects_info
    for s in subjects_info:
        agent_memory["studied_topics"][s["name"]] = []

    console.print(Rule(f"[bold cyan]🎓 Student Profile — {name}[/bold cyan]", style="cyan"))

    table = Table(
        box=box.ROUNDED,
        border_style="cyan",
        header_style="bold cyan",
        show_lines=True
    )
    table.add_column("📘 Subject",   style="bold white", min_width=24)
    table.add_column("📅 Exam Date", style="yellow",     min_width=13)
    table.add_column("⚡ Priority",  style="green",      min_width=12)
    table.add_column("⏳ Days Left", justify="center",   min_width=11)

    for s in subjects_info:
        days_left  = (date.fromisoformat(s["exam_date"]) - date.today()).days
        urgency    = "[red]🔴 High[/red]" if s["priority"] == "high" else "[yellow]🟡 Medium[/yellow]"
        days_color = "red" if days_left <= 3 else "yellow" if days_left <= 7 else "green"
        table.add_row(
            s["name"],
            s["exam_date"],
            urgency,
            f"[{days_color}]{days_left}d[/{days_color}]"
        )

    console.print(table)
    console.print(f"[bold green]✅ Profile ready for [cyan]{name}[/cyan] — {len(subjects_info)} subjects loaded.[/bold green]\n")

# ── Mark Topic as Studied ─────────────────────────────────────
def mark_studied(subject, topic):
    if subject in agent_memory["studied_topics"]:
        agent_memory["studied_topics"][subject].append(topic)
        console.print(f"  [green]✅[/green] [cyan]{subject}[/cyan] → [white]{topic}[/white]")
    else:
        console.print(f"  [red]❌ Subject '{subject}' not found.[/red]")

# ── Show Full Progress Report ─────────────────────────────────
def show_progress():
    console.print()
    console.print(Rule(f"[bold cyan]📊 Progress Report — {agent_memory['student_name']}[/bold cyan]", style="cyan"))

    for s in agent_memory["subjects"]:
        name       = s["name"]
        exam       = s["exam_date"]
        done       = agent_memory["studied_topics"].get(name, [])
        days_left  = (date.fromisoformat(exam) - date.today()).days
        days_color = "red" if days_left <= 3 else "yellow" if days_left <= 7 else "green"

        inner = Table(box=box.SIMPLE, show_header=False, padding=(0, 1))
        inner.add_column("Icon",  style="green",  min_width=3)
        inner.add_column("Topic", style="white",  min_width=32)

        if done:
            for t in done:
                inner.add_row("✅", t)
        else:
            inner.add_row("⚠️ ", "[dim]Nothing marked yet[/dim]")

        console.print(Panel(
            inner,
            title=(
                f"[bold white]{name}[/bold white]  "
                f"[dim]|[/dim]  [yellow]{exam}[/yellow]  "
                f"[dim]|[/dim]  [{days_color}]{days_left} days left[/{days_color}]"
            ),
            border_style="cyan",
            box=box.ROUNDED,
            padding=(0, 1)
        ))

# ── Session Summary ───────────────────────────────────────────
def show_summary():
    console.print()
    console.print(Rule("[bold cyan]📋 Session Summary[/bold cyan]", style="cyan"))

    table = Table(box=box.ROUNDED, border_style="cyan", header_style="bold cyan", show_lines=True)
    table.add_column("#",           style="dim",   min_width=3,  justify="center")
    table.add_column("💬 You Asked", style="white", min_width=38)
    table.add_column("🤖 Agent Said",style="cyan",  min_width=38)

    for i, log in enumerate(agent_memory["session_log"], 1):
        u = log["user"][:38]  + ("…" if len(log["user"])  > 38 else "")
        a = log["agent"][:38] + ("…" if len(log["agent"]) > 38 else "")
        table.add_row(str(i), u, a)

    console.print(table)

    total_studied = sum(len(v) for v in agent_memory["studied_topics"].values())
    console.print()
    console.print(Panel.fit(
        f"[bold green]🎓 ExamEase Session Complete![/bold green]\n\n"
        f"[dim]Student       :[/dim]  [cyan]{agent_memory['student_name']}[/cyan]\n"
        f"[dim]Interactions  :[/dim]  [white]{len(agent_memory['session_log'])}[/white]\n"
        f"[dim]Topics Marked :[/dim]  [white]{total_studied}[/white]\n\n"
        f"[bold yellow]💪 You've got this! Good luck with your exams![/bold yellow]",
        border_style="green",
        box=box.DOUBLE,
        padding=(1, 3)
    ))

console.print("[bold green]✅ All agent skills loaded and ready![/bold green]\n")

✅ All agent skills loaded and ready!

In [3]:
console.print(Rule("[bold cyan]🚀 Starting ExamEase Session[/bold cyan]", style="cyan"))
console.print()

# Setup profile
setup_student(
    name="Sayantan",
    subjects_info=[
        {"name": "Software Engineering", "exam_date": "2026-07-07", "priority": "high"},
        {"name": "HRM",                  "exam_date": "2026-07-10", "priority": "medium"},
        {"name": "DBMS",                 "exam_date": "2026-07-12", "priority": "high"},
        {"name": "Operating Systems",    "exam_date": "2026-07-15", "priority": "medium"},
    ]
)

# Skill: Schedule
ask_agent("Make me a realistic day-by-day study schedule for all my exams. I can study 3 hours per day.")

────────────────────────────────────────── 🚀 Starting ExamEase Session ───────────────────────────────────────────

────────────────────────────────────────── 🎓 Student Profile — Sayantan ──────────────────────────────────────────

╭──────────────────────────┬───────────────┬──────────────┬──────────────╮
│ 📘 Subject               │ 📅 Exam Date  │ ⚡ Priority  │ ⏳ Days Left │
├──────────────────────────┼───────────────┼──────────────┼──────────────┤
│ Software Engineering     │ 2026-07-07    │ 🔴 High      │      2d      │
├──────────────────────────┼───────────────┼──────────────┼──────────────┤
│ HRM                      │ 2026-07-10    │ 🟡 Medium    │      5d      │
├──────────────────────────┼───────────────┼──────────────┼──────────────┤
│ DBMS                     │ 2026-07-12    │ 🔴 High      │      7d      │
├──────────────────────────┼───────────────┼──────────────┼──────────────┤
│ Operating Systems        │ 2026-07-15    │ 🟡 Medium    │     10d      │
╰──────────────────────────┴───────────────┴──────────────┴──────────────╯

✅ Profile ready for Sayantan — 4 subjects loaded.

💬 You: Make me a realistic day-by-day study schedule for all my exams. I can study 3 hours per day.

⏳ ExamEase is thinking...

╭────────────────────────────────────────────────── 🤖 ExamEase ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Sayantan! I've got you covered. Given your exam dates and available study hours, I've created a realistic      │
│  day-by-day study plan for you. Here's your schedule:                                                           │
│                                                                                                                 │
│  **Study Plan (3 hours/day)**                                                                                   │
│                                                                                                                 │
│  * **July 05 (Today)**:                                                                                         │
│    • Software Engineering (1.5 hours)                                                                           │
│    • DBMS (1.5 hours)                                                                                           │
│  * **July 06**:                                                                                                 │
│    • Software Engineering (1.5 hours)                                                                           │
│    • DBMS (1.5 hours)                                                                                           │
│  * **July 07 (Exam Day)**: No study, focus on the exam!                                                         │
│  * **July 08**:                                                                                                 │
│    • HRM (1.5 hours)                                                                                            │
│    • Operating Systems (1.5 hours)                                                                              │
│  * **July 09**:                                                                                                 │
│    • HRM (1.5 hours)                                                                                            │
│    • Operating Systems (1.5 hours)                                                                              │
│  * **July 10 (Exam Day)**: No study, focus on the exam!                                                         │
│  * **July 11**:                                                                                                 │
│    • DBMS (1.5 hours)                                                                                           │
│    • Software Engineering (1.5 hours) **(Revision)**                                                            │
│  * **July 12 (Exam Day)**: No study, focus on the exam!                                                         │
│  * **July 13**:                                                                                                 │
│    • Operating Systems (1.5 hours)                                                                              │
│    • HRM (1.5 hours) **(Revision)**                                                                             │
│  * **July 14**:                                                                                                 │
│    • All subjects (quick revision, 3 hours)                                                                     │
│                                                                                                                 │
│  **July 15 (Exam Day)**: You're all set! Focus on your last exam.                                               │
│                                                                                                                 │
│  This plan allows you to distribute your study hours evenly across subjects, with some extra time for revision  │
│  before each exam. Take breaks, stay focused, and you'l

"Sayantan! I've got you covered. Given your exam dates and available study hours, I've created a realistic day-by-day study plan for you. Here's your schedule:\n\n**Study Plan (3 hours/day)**\n\n* **July 05 (Today)**: \n  • Software Engineering (1.5 hours)\n  • DBMS (1.5 hours)\n* **July 06**: \n  • Software Engineering (1.5 hours)\n  • DBMS (1.5 hours)\n* **July 07 (Exam Day)**: No study, focus on the exam!\n* **July 08**: \n  • HRM (1.5 hours)\n  • Operating Systems (1.5 hours)\n* **July 09**: \n  • HRM (1.5 hours)\n  • Operating Systems (1.5 hours)\n* **July 10 (Exam Day)**: No study, focus on the exam!\n* **July 11**: \n  • DBMS (1.5 hours)\n  • Software Engineering (1.5 hours) **(Revision)**\n* **July 12 (Exam Day)**: No study, focus on the exam!\n* **July 13**: \n  • Operating Systems (1.5 hours)\n  • HRM (1.5 hours) **(Revision)**\n* **July 14**: \n  • All subjects (quick revision, 3 hours)\n\n**July 15 (Exam Day)**: You're all set! Focus on your last exam.\n\nThis plan allows y

In [4]:
console.print(Rule("[bold cyan]🛠️  Agent Skills Demo[/bold cyan]", style="cyan"))

# Skill: Explain
ask_agent("Explain COCOMO model in simple terms. I have 30 minutes to understand it.")

# Skill: Mark Progress
console.print()
console.print(Rule("[dim]📝 Marking Studied Topics[/dim]", style="dim"))
mark_studied("Software Engineering", "SDLC Models")
mark_studied("Software Engineering", "COCOMO Model")
mark_studied("Software Engineering", "Risk Management")
mark_studied("DBMS",                 "ER Diagrams")
mark_studied("DBMS",                 "Normalization")
mark_studied("Operating Systems",    "CPU Scheduling")

# Skill: Progress Report
show_progress()

# Skill: Motivation
ask_agent("I'm really stressed about my Software Engineering exam tomorrow. Help me calm down and focus.")

# Skill: Quick Q&A
ask_agent("What is the difference between process and thread in OS? Keep it under 5 lines.")

────────────────────────────────────────────── 🛠️  Agent Skills Demo ───────────────────────────────────────────────

💬 You: Explain COCOMO model in simple terms. I have 30 minutes to understand it.

⏳ ExamEase is thinking...

╭────────────────────────────────────────────────── 🤖 ExamEase ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Sayantan, you've got this. Let's break down the COCOMO model in simple terms.                                  │
│                                                                                                                 │
│  **What is COCOMO?**                                                                                            │
│  COCOMO stands for Constructive Cost Model. It's a method used in Software Engineering to estimate the cost,    │
│  effort, and time required to complete a software project.                                                      │
│                                                                                                                 │
│  **Key Points:**                                                                                                │
│                                                                                                                 │
│  * **Three types of COCOMO models:**                                                                            │
│          + Basic ( Organic ): Used for small, simple projects                                                   │
│          + Intermediate: Used for medium-sized projects with some complexity                                    │
│          + Detailed: Used for large, complex projects                                                           │
│  * **Effort Estimation:** COCOMO estimates the effort required in **Person-Months (PM)**                        │
│  * **Factors affecting effort:**                                                                                │
│          + Project size (KLOC - Kilo Lines of Code)                                                             │
│          + Complexity                                                                                           │
│          + Team experience                                                                                      │
│          + Technology used                                                                                      │
│                                                                                                                 │
│  **Basic COCOMO Formula:**                                                                                      │
│  E = a * (KLOC) ^ b                                                                                             │
│                                                                                                                 │
│  Where:                                                                                                         │
│  E = Effort (in PM)                                                                                             │
│  a, b = Coefficients depending on the project type                                                              │
│  KLOC = Estimated project size in Kilo Lines of Code                                                            │
│                                                                                                                 │
│  **30-minute Recap:**                                                                                           │
│                                                                                                                 │
│  * COCOMO is a cost estimation model in Software Engineering                                                    │
│  * It estimates effort, cost, and time required for a project                                                   │
│  * Three types: Basic, Intermediate, and Detailed                                                               │
│  * Effort estimation is in Person-Months (PM)          

──────────────────────────────────────────── 📝 Marking Studied Topics ────────────────────────────────────────────

...

╭────────────────────────────────────────────────── 🤖 ExamEase ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Sayantan, take a deep breath. You've got this!                                                                 │
│                                                                                                                 │
│  Let's break it down:                                                                                           │
│                                                                                                                 │
│  * You've already studied key topics in Software Engineering: SDLC Models, COCOMO Model, and Risk Management.   │
│  * You have **2 days left** until the exam. Focus on revising these topics and practicing some questions.       │
│                                                                                                                 │
│  To calm your nerves, let's create a simple plan for today and tomorrow:                                        │
│                                                                                                                 │
│  **Today (Sunday, July 05):**                                                                                   │
│                                                                                                                 │
│  * 2 hours: Quick revision of Software Engineering topics (SDLC Models, COCOMO Model, Risk Management)          │
│  * 1 hour: Practice questions or past papers for Software Engineering                                           │
│                                                                                                                 │
│  **Tomorrow (Monday, July 06):**                                                                                │
│                                                                                                                 │
│  * 2 hours: Final revision and practice                                                                         │
│                                                                                                                 │
│  You've got a good foundation, Sayantan. Just relax, focus, and you'll do great!                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

💬 You: What is the difference between process and thread in OS? Keep it under 5 lines.

⏳ ExamEase is thinking...

╭────────────────────────────────────────────────── 🤖 ExamEase ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Sayantan! Focus on Operating Systems for now.                                                                  │
│                                                                                                                 │
│  * A **process** is a separate program in execution, with its own memory space.                                 │
│  * A **thread** is a lightweight process, sharing the same memory space as other threads in the same process.   │
│  * Key difference: Processes have separate memory, while threads share memory.                                  │
│                                                                                                                 │
│  You've already studied CPU Scheduling in OS. Keep going! Your exam is on July 15. What's next?                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

"Sayantan! Focus on Operating Systems for now. \n\n* A **process** is a separate program in execution, with its own memory space.\n* A **thread** is a lightweight process, sharing the same memory space as other threads in the same process.\n* Key difference: Processes have separate memory, while threads share memory.\n\nYou've already studied CPU Scheduling in OS. Keep going! Your exam is on July 15. What's next?"

In [5]:
show_summary()

─────────────────────────────────────────────── 📋 Session Summary ────────────────────────────────────────────────

╭─────┬─────────────────────────────────────────┬─────────────────────────────────────────╮
│  #  │ 💬 You Asked                            │ 🤖 Agent Said                           │
├─────┼─────────────────────────────────────────┼─────────────────────────────────────────┤
│  1  │ Make me a realistic day-by-day study s… │ Sayantan! I've got you covered. Given … │
├─────┼─────────────────────────────────────────┼─────────────────────────────────────────┤
│  2  │ Explain COCOMO model in simple terms. … │ Sayantan, you've got this. Let's break… │
├─────┼─────────────────────────────────────────┼─────────────────────────────────────────┤
│  3  │ I'm really stressed about my Software … │ Sayantan, take a deep breath. You've g… │
├─────┼─────────────────────────────────────────┼─────────────────────────────────────────┤
│  4  │ What is the difference between process… │ Sayantan! Focus on Operating Systems f… │
╰─────┴─────────────────────────────────────────┴─────────────────────────────────────────╯

╔════════════════════════════════════════════════════╗
║                                                    ║
║   🎓 ExamEase Session Complete!                    ║
║                                                    ║
║   Student       :  Sayantan                        ║
║   Interactions  :  4                               ║
║   Topics Marked :  6                               ║
║                                                    ║
║   💪 You've got this! Good luck with your exams!   ║
║                                                    ║
╚════════════════════════════════════════════════════╝